# FarmFederate - reviewer ablation suite

Runs the ablations and baselines the reviewers asked for, on the tea-pest
dataset, using FarmFederate's own models and FedAvg implementation.

| | Experiment | Answers |
|---|---|---|
| E1 | Dirichlet α sweep, **corrected vs legacy partitioner** | "single non-IID setting" + makes the non-IID claim checkable |
| E2 | Client-count sweep K | scalability |
| E3 | Anti-collapse components, federated + multimodal, two α | component contributions |
| E4 | Warm start vs cold start | initialization contribution |
| E5 | Fusion strategies × seeds | is a fusion ranking real or noise |
| E6 | Communication cost per round | cost analysis |
| E8 | FedAvg / FedProx / SCAFFOLD / FedBN / local-only, **matched settings** | comparison against other FL methods |

E7 (retrieval) is **not** in this notebook — `experiments/advisory_retrieval_eval.py`
already covers it. Run that separately rather than duplicating it.

## The point of E1

`split_data_non_iid()` behaves in two different ways. Given `labels` it splits
each class independently — a real label-skew partition. Without `labels` it
falls back to cutting contiguous Dirichlet-sized chunks, which varies client
shard **size** while leaving the label mix nearly IID.

`federated_train()`'s default path calls it *without* labels. So a sweep run
through that path varies α while the clients stay effectively IID in labels,
and the resulting "robust to non-IID" curve is flat by construction.

E1 runs both partitioners at every α and reports the realised label skew
(total-variation distance) next to the accuracy. **Report the TV column with
α** — α on its own does not establish that clients were non-IID.

## Runbook

1. **Runtime → Change runtime type → GPU.**
2. Upload in cell 3: `FarmFederate_Colab_Complete.py` (from `backend/`),
   `farm_ablation.py`, `farm_make_tables.py`, and `data_final.zip`.
3. Run the smoke cell (**~20 min**) and confirm it finishes. Do not skip it.
4. Run chunks A–D, downloading results after **each** one.
5. Build the tables, then read `paper_assets/SUMMARY.md`.

Everything is resumable: results are keyed and flushed as they complete, so
re-running a chunk skips work that already finished.

In [ ]:
#@title 1 - GPU check
import torch, subprocess
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi","--query-gpu=memory.total,memory.free",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())


In [ ]:
#@title 2 - dependencies
# The base module installs most of what it needs on import.
!pip -q install transformers 2>&1 | tail -1
print("ok")


In [ ]:
#@title 3 - upload the scripts and the data bundle
import os, sys, zipfile
from google.colab import files

NEED = ["FarmFederate_Colab_Complete.py", "farm_ablation.py",
        "farm_make_tables.py"]
missing = [f for f in NEED if not os.path.exists(f"/content/{f}")]
need_data = not os.path.isdir("/content/data_final") and \
            not os.path.exists("/content/data_final.zip")

if missing or need_data:
    print("Upload:", missing + (["data_final.zip"] if need_data else []))
    for name in files.upload():
        os.replace(name, f"/content/{name}")
        print("  saved", name)

missing = [f for f in NEED if not os.path.exists(f"/content/{f}")]
assert not missing, f"still missing: {missing}"
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
os.chdir("/content")
print("ready")


In [ ]:
#@title 4 - SMOKE TEST (~20 min). Validates every path. NOT publishable.
import importlib, farm_ablation as fa
importlib.reload(fa)

fa.main(base_py="/content/FarmFederate_Colab_Complete.py",
        tier="smoke",
        out="/content/farm_results_smoke.json")

print("\n>>> If this printed 'Done', the pipeline works. Move on to Chunk A.")


---
### Chunk A - the α sweep

Runs both partitioners at every α, so it is roughly double a plain sweep. This is the chunk that produces the headline methodological result.

In [ ]:
#@title Chunk A - E1 (alpha x partitioner)   ~2-3 h
import importlib, farm_ablation as fa
importlib.reload(fa)

fa.main(base_py="/content/FarmFederate_Colab_Complete.py",
        tier="standard",
        out="/content/farm_results.json",
        only=['E1'])

print("\n>>> CHUNK DONE - run the DOWNLOAD cell below before anything else.")


In [ ]:
#@title DOWNLOAD RESULTS - run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["farm_results.json", "farm_warmstart_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)


---
### Chunk B - scalability and the anti-collapse ablation

In [ ]:
#@title Chunk B - E2 + E3   ~2-3 h
import importlib, farm_ablation as fa
importlib.reload(fa)

fa.main(base_py="/content/FarmFederate_Colab_Complete.py",
        tier="standard",
        out="/content/farm_results.json",
        only=['E2', 'E3'])

print("\n>>> CHUNK DONE - run the DOWNLOAD cell below before anything else.")


In [ ]:
#@title DOWNLOAD RESULTS - run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["farm_results.json", "farm_warmstart_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)


---
### Chunk C - matched-setting baselines

The comparison against other federated methods. Trains four aggregation rules plus a local-only control at two α values.

In [ ]:
#@title Chunk C - E8 (FedAvg/FedProx/SCAFFOLD/FedBN/local-only)   ~2-3 h
import importlib, farm_ablation as fa
importlib.reload(fa)

fa.main(base_py="/content/FarmFederate_Colab_Complete.py",
        tier="standard",
        out="/content/farm_results.json",
        only=['E8'])

print("\n>>> CHUNK DONE - run the DOWNLOAD cell below before anything else.")


In [ ]:
#@title DOWNLOAD RESULTS - run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["farm_results.json", "farm_warmstart_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)


---
### Chunk D - initialization, fusion variance, cost

In [ ]:
#@title Chunk D - E4 + E5 + E6   ~1-2 h
import importlib, farm_ablation as fa
importlib.reload(fa)

fa.main(base_py="/content/FarmFederate_Colab_Complete.py",
        tier="standard",
        out="/content/farm_results.json",
        only=['E4', 'E5', 'E6'])

print("\n>>> CHUNK DONE - run the DOWNLOAD cell below before anything else.")


In [ ]:
#@title DOWNLOAD RESULTS - run after every chunk. Do not skip.
# Nothing in /content survives a disconnect. This is your only copy.
from google.colab import files
import os
for f in ["farm_results.json", "farm_warmstart_concat.pt"]:
    p = f"/content/{f}"
    if os.path.exists(p):
        print("downloading", f, f"({os.path.getsize(p)/1e6:.1f} MB)")
        files.download(p)
    else:
        print("not found (yet):", f)


---
### Resuming after a disconnect

In [ ]:
#@title RESUME - run this if you are continuing a previous session
# Upload BOTH farm_results.json and farm_warmstart_concat.pt (plus the
# scripts and data via cell 3). They are not interchangeable: the .json
# records which experiments finished, the .pt holds the shared warm-start
# weights E4 compares against. Without the .pt the suite retrains it
# rather than silently running a "warm" arm from random init.
from google.colab import files
import shutil, os
print("Select farm_results.json AND farm_warmstart_concat.pt:")
for name in files.upload():
    shutil.move(name, f"/content/{name}")
    print("  restored", name)
have = [f for f in os.listdir("/content") if f.endswith((".json", ".pt"))]
print("\nfiles now in /content:", have)


---
### Build the paper assets

In [ ]:
#@title Build LaTeX tables, the figure, and the readout
import importlib, sys, farm_make_tables as mt
importlib.reload(mt)

sys.argv = ["farm_make_tables.py",
            "--results", "/content/farm_results.json",
            "--outdir",  "/content/paper_assets"]
mt.main()

print("\n" + "="*70)
print(open("/content/paper_assets/SUMMARY.md").read())


In [ ]:
#@title Download paper_assets as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/paper_assets", "zip", "/content/paper_assets")
files.download("/content/paper_assets.zip")


## After it finishes

**Read `paper_assets/SUMMARY.md` before pasting anything.** It says what each
result means, including where a result argues against a claim the paper
currently makes. Three to watch:

- **E1.** If the corrected partitioner's F1 falls with α while the legacy one
  stays flat, the flat curve was an artefact. Any existing "robust to non-IID"
  claim resting on legacy-split numbers has to be restated using the corrected
  ones, with the TV column quoted alongside α.
- **E3.** If the `neither` arm does not collapse, the anti-collapse stack is not
  doing the work the paper attributes to it on this data. Weaken the claim to
  match the measurement — do not re-run until it agrees.
- **E8.** If FedProx or SCAFFOLD beats FedAvg at low α, switch the aggregator
  and say so, rather than keeping FedAvg because it is what was submitted.

One caveat that applies to every fusion number here: this data bundle has no
`crops/` directory, so image and text rows are paired only by sharing a label,
not by being observations of the same plant. `SUMMARY.md` records this as
`pairing_mode: label_matched_resample`. Fusion results should be described
accordingly.